In [ ]:
import plotly.express as px
import numpy as np
import pandas as pd
import torch
from darts.utils.statistics import plot_ccf
from sklearn.preprocessing import StandardScaler

from aare.constants import TIME
from aare.params import read_params
from aare.preparation import resample, interpolate_continuous
from aare.remote_existenz_store import RemoteExistenzStore
from aare.features.water_temp import WaterTemp
from aare.utils import to_ts, between, trunc_common

## Analysis of meteotest features

In good ol' ML fashion, I'd like to just throw them into the training and see what sticks, but for that I need to know how to clean and interpolate them.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
# logging.basicConfig(level="DEBUG")

In [ ]:
params = read_params()
store = RemoteExistenzStore()

In [ ]:
df = store.query(
    # ANYTIME,
    "2004-02-01",  # new start of SMN data, thanks to new import of opendata release :O could adjust train_split if we want
    # (params["split"]["train_split"]),
    # (params["split"]["train_split"], params["split"]["val_split"]),
    [
        "hydro/temperature:mean_1h@bern",
        "smn/tt:mean_1h@bern",  # air temp °C
        "smn/ss:sum_1h@bern",  # sunshine duration min
        "smn/rr:sum_1h@bern",  # rain mm
        "smn/rh:mean_1h@bern",  # relative humidity %
        "smn/dd:mean_1h@bern",  # wind direction °
        "smn/ff:mean_1h@bern",  # wind speed km/h
        "smn/dd:mean_1h@thun",  # thun is more interesting becaues of the lake
        "smn/ff:mean_1h@thun",
    ],
)
df

In [ ]:
df = resample(df)
df = WaterTemp("bern").cleanup(df)
df

In [ ]:
df.describe().T

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
def plt(df, *cols: str, sec=None, point_size=2, line=False):
    func = px.line if line else px.scatter
    return func(df, x=TIME, y=list(cols)).update_traces(marker={"size": point_size})

## Sunshine (ss) and air temp (tt)

- Already done, see 07 deeper questions

## Relative Humidity (RH)

- Percentage, so 0-100, but never drop below 18% in these measurements so I don't think we need a (reasonable) bound at 10 or 15% and can just use 0.
- No obvious outliers or measurements errors visible, amazingly clean data.
- Looking at the gaps, it seems like linear interpolation would work fine 3, maybe even 5 hours
- The largest gap here is 7 hours and could in this specific case also be filled by linear interpolation, but cubic look great too
- Playing around with it, it seems that even for gaps of size 10 cubic interpolation can fail pretty badly, but to be honest, it's probably fine.
- If we wanted actually good imputation, we'd need to go for pypots. For now, let's just use linear=3, cubic=10 and hope for the best :)

In [ ]:
plt(df, "temperature_bern", "rh_bern")

In [ ]:
to_ts(df, "rh_bern").gaps().sort_values("gap_size", ascending=False)

In [ ]:
df_t = df.copy()
df_t.loc[between(df_t, "2025-08-01T08:00:00", "2025-08-01T23:59:00"), "rh_bern"] = np.nan
df_t.loc[between(df_t, "2025-08-02T18:00:00", "2025-08-03T03:59:00"), "rh_bern"] = np.nan
to_ts(df_t, "rh_bern").gaps().sort_values("gap_size", ascending=False)

In [ ]:
df_i = interpolate_continuous(
    df_t, linear_gap_bound=3, cubic_gap_bound=20, median_gap_bound=None, drop_filled=True, columns="rh_bern"
)
df_t["rh_bern_filled"] = df_i["rh_bern"]
df_t["rh_bern"] = df["rh_bern"]

plt(df_t, "temperature_bern", "rh_bern_filled", "rh_bern")

## Rainfall (RR)

- Millimeter sum, so must be positive, but no upper bound
- Data looks super clean, this is amazing.
- Definitely cannot find outliers with diff based, there can always be sudden showers that last less than an hour
- Visual analysis leads me to believe that gaps can be handled very similarly to sunshine, since it's also just a sum with occasional peaks.
  RR is more "sparse" with just occasional spikes instead of longer periods with sunshine, so it could make sense to tune the median window a bit. I'll take the same as sunshine for a start (10) because it's muuuch harder to fill this in correctly than to correctly imputate a slow temperature for example. Luckily the largest gap is 7 hours all of the data.

In [ ]:
plt(df, "temperature_bern", "rr_bern")

In [ ]:
df_i = interpolate_continuous(df, None, None, median_gap_bound=7, drop_filled=True, columns="rr_bern")
plt(df_i, "temperature_bern", "rr_bern")

In [ ]:
to_ts(df, "rr_bern").gaps().sort_values("gap_size", ascending=False)

## Wind (DD and FF)

- can be transformed into x and y strength with sin and cos (easier for models to use)
- degree (dd_bern) has range [0, 360], while magnitude is unbounded but must be positive
- After transformation into x and y components, there are no bounds anymore
- Really weird constant dd_bern before a gap in January 2009, not sure how to clean
  - remove_period(df, "01.01.2009T12:00:00Z", "06.01.2009T00:00:00Z", self.field.name)
  - shortly after, there are some more shorter periods where it's almost constant, but those could be realistic
- the strength can be interpolated linearly up to 5 hours I'd say. Up to 10 hours might work with cubic
- the direction of the wind seems impossible to accuratly interpolate. to avoid a ton of missing data when there is just 1-3 hours missing, I'd say 3-5 hours can be filled in linearly. Since those are coupled, there's no need to think about ff anymore since data that is interpolated for ff but not for dd will be cut anyway.
- doing the interpolation on the transformed x and y components seems like it would be even harder, so let's not do that

In [ ]:
for l in ["bern", "thun"]:
    df[f"wind_y_{l}"] = np.sin(df[f"dd_{l}"]) * df[f"ff_{l}"]
    df[f"wind_x_{l}"] = np.cos(df[f"dd_{l}"]) * df[f"ff_{l}"]

In [ ]:
plt(df, "temperature_bern", "ff_bern", "dd_bern")

In [ ]:
plt(df, "temperature_bern", "wind_x_bern", "wind_y_bern", line=True)

In [ ]:
to_ts(df, "ff_bern").gaps().sort_values("gap_size", ascending=False)

#### Playing around with wind

Tried to confirm the theory that the water cools very quickly if the wind blows south-east in Thun.

In [ ]:
to_ts(df, "ff_thun").gaps().sort_values("gap_size", ascending=False)

In [ ]:
temp = df.set_index(TIME)
scaled_raw = StandardScaler().fit_transform(temp)
scaled = pd.DataFrame(scaled_raw, index=temp.index, columns=temp.columns).reset_index()
scaled

In [ ]:
temp_bern = to_ts(df, "temperature_bern")
ff_bern = to_ts(df, "ff_bern")
dd_bern = to_ts(df, "dd_bern")
ff_thun = to_ts(df, "ff_thun")
dd_thun = to_ts(df, "dd_thun")

In [ ]:
temp_bern, ff_bern, dd_bern, ff_thun, dd_thun = trunc_common(temp_bern, ff_bern, dd_bern, ff_thun, dd_thun)

In [ ]:
temp_bern.start_time(), temp_bern.end_time()

In [ ]:
plot_ccf(ff_bern, ff_thun)

In [ ]:
plot_ccf(ff_thun, ff_bern)

In [ ]:
wind_x_bern = to_ts(df, "wind_x_bern")
wind_y_bern = to_ts(df, "wind_y_bern")
wind_x_thun = to_ts(df, "wind_x_thun")
wind_y_thun = to_ts(df, "wind_y_thun")

_, wind_x_bern, wind_y_bern, wind_x_thun, wind_y_thun = trunc_common(
    temp_bern, wind_x_bern, wind_y_bern, wind_x_thun, wind_y_thun
)

In [ ]:
plot_ccf(wind_x_bern, wind_x_thun)

In [ ]:
plot_ccf(wind_x_thun, wind_x_bern)

In [ ]:
plot_ccf(wind_y_bern, wind_y_thun)

In [ ]:
plot_ccf(wind_y_thun, wind_y_bern)

seems like the wind in berne is ahead of thun, which makes sense because the weather comes from the west. but still, only ~0.6 correlation for ff, so the wind in thun can definitely be different that what we measure in bern (=it _is_ a bummer that thun is only available from 2013 and bern from 2004 on). For other wind features it's even lower, supporting that hypothesis.

In [ ]:
for_ccf = df.copy()
for_ccf.loc[(df["dd_bern"] < 120) | (df["dd_bern"] > 150), "dd_bern"] = np.nan
temp_bern = to_ts(for_ccf, "temperature_bern")
ff_bern = to_ts(for_ccf, "ff_bern")
temp_bern, ff_bern = trunc_common(temp_bern, ff_bern)

plot_ccf(temp_bern, ff_bern)

In [ ]:
df["temperature_bern"].diff().mean()

In [ ]:
lag = 3

In [ ]:
df["temperature_bern"].diff().shift(lag)[(df["dd_bern"] > 120) & (df["dd_bern"] < 150)].mean()

In [ ]:
df["temperature_bern"].diff().shift(lag)[(df["dd_bern"] < 120) | (df["dd_bern"] > 150)].mean()

In [ ]:
df["temperature_bern"].diff().shift(lag)[
    (df["wind_x_bern"] > 10) & ((df["wind_y_bern"] + df["wind_x_bern"]).abs() < 1)
].mean()

In [ ]:
df["temperature_bern"].diff().shift(lag)[
    (df["wind_x_bern"] > 10) & ((df["wind_y_bern"] + df["wind_x_bern"]).abs() > 1)
].mean()